# 03 – Model Training
This notebook trains the selected models and persists *all* artefacts. It delegates **all heavy‑lifting** to the unified `src.training.run_training` helper.

In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2
Configuration: {'general': {'run_name': 'experiment_with_4_classes', 'seed': 42}, 'dataset': {'split_type': 'test', 'n_classes': 5, 'n_samples_per_class': 10}, 'paths': {'data_exploration_dir': 'output/experiment_with_4_classes/data_exploration', 'embeddings_dir': 'output/experiment_with_4_classes/embeddings', 'models_dir': 'output/experiment_with_4_classes/models', 'predictions_dir': 'output/experiment_with_4_classes/predictions', 'results_dir': 'output/experiment_with_4_classes/results'}, 'model': {'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False}, 'training': {'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3'}, 'evaluation': {'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}}

=== Configuration Variables ===

[DATASET]
  DATASET_N_CLASSES: 5
  

In [2]:
from config.notebook_setup import *

# Your helpers (get_dataset, model classes…) live in *src*
from src.datasets.dataset import get_dataset
from src.training import run_training

In [3]:


# Load dataset
# X_train, y_train, _, _, _ = get_dataset(split_type="standard", n_classes=N_CLASSES)
X_train, y_train, X_test, y_test, classes = get_dataset(
    split_type=DATASET_SPLIT_TYPE,
    n_classes=N_CLASSES,
    n_samples_per_class=N_SAMPLES_PER_CLASS,

)

print(f"Loaded {len(X_train)} training documents with {N_CLASSES} classes")


INFO | Loading Reuters dataset with configuration:
INFO |   - Split type: test
INFO |   - Number of classes: 5
INFO |   - Samples per class: all available
INFO |   - Random seed: None
INFO | Loading small test dataset with all available samples per class across 5 classes
INFO | Selected classes: earn, acq, crude, interest, money-fx
INFO |   - Class 'earn': 2748 train, 1178 test
INFO |   - Class 'acq': 1658 train, 711 test
INFO |   - Class 'crude': 386 train, 166 test
INFO |   - Class 'interest': 317 train, 136 test
INFO |   - Class 'money-fx': 253 train, 109 test


Loaded 5362 training documents with 5 classes


In [4]:
import os
from src.datasets.dataset import get_dataset
from src.algorithms.naive_bayes import NaiveBayesClassifier
from src.algorithms.linear_svm import LinearSVMClassifier, LinearSVMBigrams
from src.algorithms.transformer_logreg import TransformerLogReg
from src.rag import load_kmajority, load_centroid, load_llm
from src.rag.adapter_sklearn import RagSklearnAdapter
# from src.rag.rag_hybrid import RagHybrid
from src.embeddings.openai_embedder import OpenAIEmbedder
from src.rag.vector_store import VectorStore


/home/marcmaceira/projects/reuters-rag-classifier_clean_v2/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO | Loading faiss with AVX2 support.
INFO | Successfully loaded faiss with AVX2 support.
INFO | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.


In [5]:
# -------------------------------------------------------------
# 2. Define models
# -------------------------------------------------------------
models = {
    'Naive Bayes':       NaiveBayesClassifier(),
    'Linear SVM':        LinearSVMClassifier(),
    'TF-IDF bigrams + SVM': LinearSVMBigrams(),
    'MiniLM + LogReg':   TransformerLogReg(),
    # RAG variants all take an embedder under the hood:
    'RAG-kMajority':     RagSklearnAdapter(load_kmajority(top_k=5, use_openai=False)),
    'RAG-CentroidNN':    RagSklearnAdapter(load_centroid()),
    # For the LLM‐backed RAG we also pass the same embedder plus your LLM choice:
    'RAG-LLM (OpenAI-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            use_openai=True  # Add this parameter to use the OpenAI index
        )
    ),
    # Add the local embeddings variant:
    'RAG-LLM (local-embeddings)': RagSklearnAdapter(
        load_llm(
            top_k=5,
            model="gpt-4o-mini",
            use_openai=False,  # Explicitly specify to use local index
            embedder=lambda texts: VectorStore.embed("sentence-transformers/all-MiniLM-L6-v2", texts)
        )
    ),
    # Add the hybrid RAG classifier:
    # 'RAG-Hybrid':        RagHybrid.from_default(top_k=5),
}


INFO | Use pytorch device_name: cpu
INFO | Load pretrained SentenceTransformer: all-MiniLM-L6-v2
INFO | Loading SentenceTransformer retriever
INFO | Using index: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_4_classes/embeddings/sbert/index.faiss, meta: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_4_classes/embeddings/sbert/meta.jsonl
INFO | Default retriever loaded in 0.09 seconds
INFO | Loading SentenceTransformer retriever
INFO | Using index: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_4_classes/embeddings/sbert/index.faiss, meta: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_4_classes/embeddings/sbert/meta.jsonl
INFO | Default retriever loaded in 0.05 seconds
INFO | Loading OpenAI retriever
INFO | Using index: /home/marcmaceira/projects/reuters-rag-classifier_clean_v2/output/experiment_with_4_classes/embeddings/openai/index.faiss, meta: 

In [6]:

# -------------------------------------------------------------
# 3. Train + persist
# -------------------------------------------------------------
trained = run_training(
    models,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    output_dir=MODELS_DIR,
)


[run_training] Fitting Naive Bayes…
Error training model Naive Bayes: y must be a numpy array


RuntimeError: Failed to train model Naive Bayes: y must be a numpy array